In [36]:
import pandas as pd
import numpy as np

In [37]:
train_df = pd.read_csv("../data/raw/train.csv")
test_df = pd.read_csv("../data/raw/test.csv")

train_df.shape, test_df.shape


((1460, 81), (1459, 80))

In [38]:
neigh_price = train_df.groupby("Neighborhood")["SalePrice"].median()

In [39]:
y = train_df["SalePrice"]
y_log = np.log1p(y)
X = train_df.drop(columns=["SalePrice"])

In [40]:
numeric_cols = X.select_dtypes(include=["int64", "float64"]).columns
categorical_cols = X.select_dtypes(include=["object"]).columns

numeric_cols.shape, categorical_cols.shape

((37,), (43,))

# Step 5A — Numeric missing values → fill with median

In [41]:
X_median = X[numeric_cols].median()

X[numeric_cols] = X[numeric_cols].fillna(X_median)
test_df[numeric_cols] = test_df[numeric_cols].fillna(X_median)

# Step 5B — Categorical missing values → fill with "None"

In [42]:
X[categorical_cols] = X[categorical_cols].fillna("None")
test_df[categorical_cols] = test_df[categorical_cols].fillna("None")

# Step 5C — Confirm no missing values remain

In [43]:
X.isnull().sum().sum(), test_df.isnull().sum().sum()

(np.int64(0), np.int64(0))

In [44]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

In [45]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ]
)

preprocessor.fit(X)

X_processed = preprocessor.transform(X)
test_processed = preprocessor.transform(test_df)

In [46]:
feature_names = list(numeric_cols) + list(
    preprocessor.named_transformers_["cat"].get_feature_names_out(categorical_cols)
)

X_processed_df = pd.DataFrame(X_processed.toarray(), columns=feature_names)
test_processed_df = pd.DataFrame(test_processed.toarray(), columns=feature_names)

X_processed_df.shape, test_processed_df.shape

((1460, 304), (1459, 304))

### Total Bathrooms (TotalBath)

In [47]:
X["TotalSF"] = X["TotalBsmtSF"] + X["1stFlrSF"] + X["2ndFlrSF"]
test_df["TotalSF"] = test_df["TotalBsmtSF"] + test_df["1stFlrSF"] + test_df["2ndFlrSF"]

### House Age and Remodel Age

In [48]:
X["HouseAge"] = X["YrSold"] - X["YearBuilt"]
test_df["HouseAge"] = test_df["YrSold"] - test_df["YearBuilt"]

X["RemodAge"] = X["YrSold"] - X["YearRemodAdd"]
test_df["RemodAge"] = test_df["YrSold"] - test_df["YearRemodAdd"]

X["IsRemodeled"] = (X["YearBuilt"] != X["YearRemodAdd"]).astype(int)
test_df["IsRemodeled"] = (test_df["YearBuilt"] != test_df["YearRemodAdd"]).astype(int)

### Porch Area

In [49]:
porch_cols = ['WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch']

X['TotalPorchSF'] = X[porch_cols].sum(axis=1)
test_df['TotalPorchSF'] = test_df[porch_cols].sum(axis=1)

### HasGarage, HasPool, HasFireplace, HasBasement

In [50]:
X['HasGarage'] = (X['GarageArea'] > 0).astype(int)
test_df['HasGarage'] = (test_df['GarageArea'] > 0).astype(int)

X['HasPool'] = (X['PoolArea'] > 0).astype(int)
test_df['HasPool'] = (test_df['PoolArea'] > 0).astype(int)

X['HasFireplace'] = (X['Fireplaces'] > 0).astype(int)
test_df['HasFireplace'] = (test_df['Fireplaces'] > 0).astype(int)

X['HasBsmt'] = (X['TotalBsmtSF'] > 0).astype(int)
test_df['HasBsmt'] = (test_df['TotalBsmtSF'] > 0).astype(int)

# Neighborhood median price

In [51]:


X["NeighborhoodMedian"] = X["Neighborhood"].map(neigh_price)
test_df["NeighborhoodMedian"] = test_df["Neighborhood"].map(neigh_price)

### Re-run missing value handling

In [52]:
X[numeric_cols] = X[numeric_cols].fillna(X[numeric_cols].median())
test_df[numeric_cols] = test_df[numeric_cols].fillna(X[numeric_cols].median())

# Step 7G — Update numeric and categorical column lists

In [53]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = X.select_dtypes(include=['object']).columns

# Re-fit the preprocessor

In [54]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ]
)

preprocessor.fit(X)

ColumnTransformer(transformers=[('num', 'passthrough',
                                 Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvG...
       'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation',
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])

# Re-transform and re-save processed data

In [55]:
X_processed = preprocessor.transform(X)
test_processed = preprocessor.transform(test_df)

# Convert to DataFrame
feature_names = (
    list(numeric_cols) +
    list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
)

X_processed_df = pd.DataFrame(X_processed.toarray(), columns=feature_names)
test_processed_df = pd.DataFrame(test_processed.toarray(), columns=feature_names)

# Save
X_processed_df.to_csv("../data/processed/train_processed.csv", index=False)
test_processed_df.to_csv("../data/processed/test_processed.csv", index=False)
y.to_csv("../data/processed/y.csv", index=False)
y_log.to_csv("../data/processed/y_log.csv", index=False)

In [56]:
X_processed_df.shape

(1460, 314)

### Check that test and train have identical columns

In [57]:
set(X_processed_df.columns) == set(test_processed_df.columns)

True